[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SekyungHan/ai-power-textbook-labs/blob/master/labs/ch07_research_pipeline.ipynb)

# Ch 7: 에이전트 기반 연구 파이프라인

In [ ]:
# Ch07: 에이전트 기반 연구 파이프라인
import sys, subprocess
if 'google.colab' in sys.modules:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                          'pandapower', 'pypsa'])
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                          'koreanize-matplotlib'])
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

np.random.seed(42)

try:
    sys.path.insert(0, '..')
    from utils.style import set_style, COLORS
    set_style()
except:
    # 한글 폰트 fallback (utils.style 없는 환경)
    import platform as _pf
    _sys = _pf.system()
    if _sys == 'Windows':
        plt.rcParams['font.family'] = 'Malgun Gothic'
    elif _sys == 'Darwin':
        plt.rcParams['font.family'] = 'AppleGothic'
    else:
        try:
            import koreanize_matplotlib
        except ImportError:
            plt.rcParams['font.family'] = 'NanumGothic'
    plt.rcParams['axes.unicode_minus'] = False


## 1. 경제 급전 (Economic Dispatch)

In [ ]:
from scipy.optimize import minimize

cost_coeff = np.array([
    [0.0430, 20.0, 0],
    [0.0250, 15.0, 0],
    [0.0100, 30.0, 0],
    [0.0120, 35.0, 0],
    [0.0080, 10.0, 0],
])
P_min = np.array([0, 0, 0, 0, 0])
P_max = np.array([332, 140, 100, 100, 100])
P_demand = 259.0

def total_cost(P):
    return sum(cost_coeff[i,0]*P[i]**2 + cost_coeff[i,1]*P[i] + cost_coeff[i,2] for i in range(5))

constraints = {'type': 'eq', 'fun': lambda P: sum(P) - P_demand}
bounds = list(zip(P_min, P_max))

result = minimize(total_cost, x0=[50]*5, method='SLSQP', bounds=bounds, constraints=constraints)
print(f"최적 발전량: {result.x.round(1)} MW")
print(f"최소 비용: {result.fun:.1f} $/h")

# 시각화
fig, ax = plt.subplots(figsize=(8, 5))
gen_names = ['G1\n(Slack)', 'G2', 'G3', 'G6', 'G8']
colors = ['#1B7A8A' if p > 0 else '#cccccc' for p in result.x]
ax.bar(gen_names, result.x, color=colors)
ax.set_ylabel('출력 (MW)')
ax.set_title(f'경제급전 결과 — 총 비용: {result.fun:.1f} $/h')
for i, v in enumerate(result.x):
    if v > 0:
        ax.text(i, v + 2, f'{v:.1f}', ha='center', fontsize=9)
plt.tight_layout()
plt.show()

## 2. pandapower Monte Carlo 시뮬레이션

In [ ]:
import pandapower as pp
import pandapower.networks as pn

n_scenarios = 100
results = []

for i in range(n_scenarios):
    net = pn.case14()
    load_scale = 1.0 + np.random.uniform(-0.2, 0.2, len(net.load))
    net.load['p_mw'] *= load_scale
    net.load['q_mvar'] *= load_scale
    try:
        pp.runpp(net, verbose=False)
        results.append({
            'scenario': i,
            'v_min': net.res_bus['vm_pu'].min(),
            'v_max': net.res_bus['vm_pu'].max(),
            'total_loss': net.res_line['pl_mw'].sum(),
            'max_loading': net.res_line['loading_percent'].max(),
        })
    except:
        results.append({'scenario': i, 'converged': False})

df = pd.DataFrame(results).dropna()
print(f"수렴 시나리오: {len(df)}/{n_scenarios}")
print(f"전압 범위: {df['v_min'].min():.4f} ~ {df['v_max'].max():.4f} p.u.")
print(f"최대 선로 부하율: {df['max_loading'].max():.1f}%")

## 2.1 시뮬레이션 결과 자동 검증

교재의 3단계 검증 코드: 수렴 → 전력균형 → 전압범위/과부하 확인

In [ ]:
def validate_results(net):
    """3단계 자동 검증"""
    issues = []

    # 문법: 수렴 확인
    if not net['converged']:
        issues.append("CRITICAL: 조류 계산 미수렴")
        return issues

    # 논리: 전력 균형
    p_gen = net.res_gen['p_mw'].sum() + net.res_ext_grid['p_mw'].sum()
    p_load = net.res_load['p_mw'].sum()
    p_loss = net.res_line['pl_mw'].sum()
    balance_err = abs(p_gen - p_load - p_loss)
    if balance_err > 0.1:
        issues.append(f"WARNING: 전력 불균형 {balance_err:.3f} MW")

    # 물리: 전압 범위 (PV/slack 모선 제외)
    pv_buses = set(net.gen['bus'].tolist() + net.ext_grid['bus'].tolist())
    pq_vm = net.res_bus.drop(index=pv_buses, errors='ignore')['vm_pu']
    v_min, v_max = pq_vm.min(), pq_vm.max()
    if v_min < 0.9 or v_max > 1.1:
        issues.append(f"WARNING: 전압 이상 ({v_min:.3f}~{v_max:.3f} p.u.)")

    # 물리: 선로 과부하
    overloaded = net.res_line[net.res_line['loading_percent'] > 100]
    if len(overloaded) > 0:
        issues.append(f"WARNING: {len(overloaded)}개 선로 과부하")

    return issues if issues else ["PASS: 모든 검증 통과"]

# 테스트: IEEE 14-bus에서 검증 실행
net_test = pn.case14()
pp.runpp(net_test, verbose=False)
print("=== validate_results 테스트 ===")
for msg in validate_results(net_test):
    print(f"  {msg}")

## 3. PyPSA 24시간 OPF

In [ ]:
import pypsa

n = pypsa.Network()
n.set_snapshots(range(24))

n.add("Bus", "bus_slack", v_nom=345)
n.add("Bus", "bus_load", v_nom=345)
n.add("Bus", "bus_wind", v_nom=345)

n.add("Generator", "gen_coal", bus="bus_slack", p_nom=500, marginal_cost=30, committable=False)

wind_cf = 0.3 + 0.2 * np.sin(np.linspace(0, 2*np.pi, 24))
n.add("Generator", "gen_wind", bus="bus_wind", p_nom=200, marginal_cost=0, p_max_pu=wind_cf, carrier="wind")

load_pattern = 300 + 100 * np.sin(np.linspace(0, 2*np.pi, 24))
n.add("Load", "load_1", bus="bus_load", p_set=load_pattern)

n.add("Line", "line_1", bus0="bus_slack", bus1="bus_load", s_nom=400, x=0.1, r=0.01)
n.add("Line", "line_2", bus0="bus_wind", bus1="bus_load", s_nom=300, x=0.15, r=0.02)

n.optimize(solver_name="highs")
print(f"총 비용: {n.objective:.0f} $")

# 발전 스택 시각화
fig, ax = plt.subplots(figsize=(10, 5))
coal = n.generators_t.p['gen_coal']
wind = n.generators_t.p['gen_wind']
ax.fill_between(range(24), 0, coal, alpha=0.7, label='석탄', color='#D4984A')
ax.fill_between(range(24), coal, coal + wind, alpha=0.7, label='풍력', color='#1B7A8A')
ax.plot(range(24), load_pattern, 'k-', lw=2, label='부하')
ax.set_xlabel('시간')
ax.set_ylabel('출력 (MW)')
ax.set_title('24시간 발전 스택')
ax.legend()
plt.tight_layout()
plt.show()

## 4. N-1 Contingency 분석

> **참고: IEEE 14-bus 전압 특성**  
> IEEE 14-bus 원본 데이터에서 일부 PQ 모선(bus 6, 8, 9~12)의 전압이 기본 케이스에서
> 이미 1.05 p.u.를 약간 초과합니다 (예: bus 6 = 1.062 p.u.).  
> 따라서 N-1 분석 시 엄격한 (0.95, 1.05) 범위를 적용하면 거의 모든 contingency에서
> 위반이 발생합니다. 본 실습에서는 이 특성을 고려하여 (0.94, 1.06) 범위를 사용합니다.

### IEEE 14-bus N-1 분석 시 참고사항

> **참고**: IEEE 14-bus 원본 데이터에서 일부 PQ 버스(6, 8, 9~12번)의 전압이
> 기본 케이스(N-0)에서 이미 1.05 p.u.를 약간 초과합니다.
> 따라서 `v_limits=(0.94, 1.06)` 기준으로도 대부분의 contingency에서
> `v_violation=True`가 발생할 수 있습니다.
> 이는 코드 버그가 아니라 **IEEE 14-bus 표준 데이터의 특성**입니다.
> 대규모 시스템(IEEE 118-bus 등)에서는 N-1 위반이 더 의미 있게 나타납니다.

In [ ]:
def run_n1_analysis(net_func, v_limits=(0.94, 1.06), loading_limit=100):
    """N-1 contingency 분석 자동 수행"""
    results = []
    net = net_func()
    pp.runpp(net, verbose=False)
    
    for idx in net.line.index:
        net = net_func()
        net.line.at[idx, 'in_service'] = False
        try:
            pp.runpp(net, verbose=False)
            # PV/slack 모선 제외 (전압 제어 모선)
            pv_buses = set(net.gen['bus'].tolist() + net.ext_grid['bus'].tolist())
            pq_vm = net.res_bus.drop(index=pv_buses, errors='ignore')['vm_pu']
            v_min = pq_vm.min()
            v_max = pq_vm.max()
            max_load = net.res_line['loading_percent'].max()
            results.append({
                'type': 'line', 'index': idx,
                'converged': True,
                'v_min': round(v_min, 4), 'v_max': round(v_max, 4),
                'max_loading': round(max_load, 2),
                # TODO: v_violation과 loading_violation 판정을 추가하세요
                'v_violation': None,  # <-- 구현하세요
                'loading_violation': None,  # <-- 구현하세요
            })
        except:
            results.append({'type': 'line', 'index': idx, 'converged': False,
                           'v_violation': True, 'loading_violation': True})
    
    return pd.DataFrame(results)

df_n1 = run_n1_analysis(pn.case14)
print(f"총 contingency: {len(df_n1)}")
print(df_n1[['index', 'v_min', 'v_max', 'max_loading']].to_string())

## 5. 민감도 분석 시나리오 매트릭스

In [ ]:
import itertools
from scipy.stats import qmc

params = {
    'load_scale': np.linspace(0.8, 1.2, 5),
    'wind_penetration': [0.1, 0.2, 0.3, 0.4],
    'gas_price': [20, 30, 40],
}

keys = list(params.keys())
combos = list(itertools.product(*params.values()))
scenarios = pd.DataFrame(combos, columns=keys)
print(f"Full factorial 시나리오 수: {len(scenarios)}")
print(scenarios.head(10))

# LHS 데모
sampler = qmc.LatinHypercube(d=len(keys), seed=42)
sample = sampler.random(n=50)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
random_pts = np.random.rand(50, 2)
axes[0].scatter(random_pts[:, 0], random_pts[:, 1], s=20, color='#C75C3A')
axes[0].set_title('단순 랜덤 샘플링')
axes[0].set_xlabel('파라미터 1')
axes[0].set_ylabel('파라미터 2')

axes[1].scatter(sample[:, 0], sample[:, 1], s=20, color='#1B7A8A')
axes[1].set_title('Latin Hypercube Sampling')
axes[1].set_xlabel('파라미터 1')
axes[1].set_ylabel('파라미터 2')

plt.tight_layout()
plt.show()

## 6. Demo 7.2b: IEEE 39-bus + 재생에너지 워크숍

IEEE 39-bus 시스템에 풍력(모선 25, 26, 29) + 태양광(모선 20, 23)을 추가하고, 침투율별 시나리오를 구성합니다.

In [ ]:
import pypsa
import pandapower as pp
import pandapower.networks as pn

# IEEE 39-bus 기본 시스템 로드
net_pp = pn.case39()
pp.runpp(net_pp, verbose=False)
total_gen_capacity = net_pp.gen['max_p_mw'].sum() + net_pp.ext_grid['max_p_mw'].sum()

def build_39bus_pypsa(net_pp, penetration=0.0):
    """pandapower case39 -> PyPSA transport model + 재생에너지"""
    n = pypsa.Network()
    n.set_snapshots(range(24))

    # 1) 버스
    for idx in net_pp.bus.index:
        n.add("Bus", f"bus_{idx}")

    # 2) 전력 수송 (Link: transport model, 양방향)
    #    교육 목적: KVL 제약 없이 경제 급전 효과에 집중
    for idx, line in net_pp.line.iterrows():
        vn = net_pp.bus.at[line['from_bus'], 'vn_kv']
        s_nom = line['max_i_ka'] * vn * 1.732
        n.add("Link", f"link_{idx}",
              bus0=f"bus_{line['from_bus']}",
              bus1=f"bus_{line['to_bus']}",
              p_nom=max(s_nom, 200),
              p_min_pu=-1)  # 양방향

    for idx, trafo in net_pp.trafo.iterrows():
        n.add("Link", f"trafo_{idx}",
              bus0=f"bus_{trafo['hv_bus']}",
              bus1=f"bus_{trafo['lv_bus']}",
              p_nom=trafo['sn_mva'] * 1.5,
              p_min_pu=-1)

    # 3) 기존 발전기 (pandapower 용량 + 한계비용)
    marginal_costs = [20.0, 15.0, 10.0, 12.0, 14.0, 16.0, 13.0, 11.0, 18.0]
    for i, row in net_pp.gen.iterrows():
        mc = marginal_costs[i] if i < len(marginal_costs) else 15.0
        n.add("Generator", f"gen_{i}",
              bus=f"bus_{int(row['bus'])}",
              p_nom=row['max_p_mw'],
              marginal_cost=mc,
              carrier="conventional")

    for i, row in net_pp.ext_grid.iterrows():
        n.add("Generator", f"slack_{i}",
              bus=f"bus_{int(row['bus'])}",
              p_nom=net_pp.ext_grid.at[i, 'max_p_mw'],
              marginal_cost=17.0,
              carrier="conventional")

    # 4) 부하 (24시간 프로파일)
    hours = np.arange(24)
    load_profile = 0.7 + 0.3 * np.sin((hours - 4) * np.pi / 12)
    load_profile = np.clip(load_profile, 0.6, 1.0)

    for idx, load in net_pp.load.iterrows():
        n.add("Load", f"load_{idx}",
              bus=f"bus_{int(load['bus'])}",
              p_set=load['p_mw'] * load_profile)

    # 5) 재생에너지
    if penetration > 0:
        re_capacity = total_gen_capacity * penetration
        wind_cap = re_capacity * 0.6
        solar_cap = re_capacity * 0.4

        wind_cf = np.clip(
            0.3 + 0.15 * np.sin(np.linspace(0, 2*np.pi, 24)), 0.05, 0.6)
        for bus_id in [25, 26, 29]:
            n.add("Generator", f"wind_{bus_id}",
                  bus=f"bus_{bus_id}",
                  p_nom=wind_cap / 3,
                  marginal_cost=0,
                  p_max_pu=wind_cf,
                  carrier="wind")

        solar_cf = np.maximum(0, np.sin((hours - 6) * np.pi / 12))
        for bus_id in [20, 23]:
            n.add("Generator", f"solar_{bus_id}",
                  bus=f"bus_{bus_id}",
                  p_nom=solar_cap / 2,
                  marginal_cost=0,
                  p_max_pu=solar_cf,
                  carrier="solar")

    return n

# 침투율별 시나리오 실행
import warnings; warnings.filterwarnings('ignore')
penetrations = [0.0, 0.1, 0.2, 0.3, 0.5]
results_39 = []

print(f"총 기존 발전 용량: {total_gen_capacity:.0f} MW")
print(f"총 부하 (피크): {net_pp.load['p_mw'].sum():.0f} MW")
print(f"시나리오: 침투율 {penetrations}\n")

for pen in penetrations:
    n39 = build_39bus_pypsa(net_pp, pen)
    try:
        status = n39.optimize(
            solver_name="highs",
            solver_options={"output_flag": False})
        if status[1] == "optimal":
            total_cost = n39.objective
            re_gen = 0
            for carrier in ["wind", "solar"]:
                gens = n39.generators[n39.generators.carrier == carrier].index
                if len(gens) > 0:
                    re_gen += n39.generators_t.p[gens].sum().sum()
            total_gen = n39.generators_t.p.sum().sum()
            re_share = re_gen / total_gen * 100 if total_gen > 0 else 0
            results_39.append({
                'penetration': pen, 'total_cost': total_cost,
                're_share': re_share, 'status': 'optimal'})
            print(f"  침투율 {pen:.0%}: 비용 ${total_cost:,.0f}, RE 발전비중 {re_share:.1f}%")
        else:
            results_39.append({'penetration': pen, 'status': str(status)})
            print(f"  침투율 {pen:.0%}: {status}")
    except Exception as e:
        results_39.append({'penetration': pen, 'status': f'error: {e}'})
        print(f"  침투율 {pen:.0%}: 오류 -- {e}")

# 결과 시각화
opt = [r for r in results_39 if r.get('status') == 'optimal']
if len(opt) >= 2:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
    pens = [r['penetration'] * 100 for r in opt]
    costs = [r['total_cost'] for r in opt]
    shares = [r['re_share'] for r in opt]

    ax1.plot(pens, costs, 'o-', color='#1B7A8A', linewidth=2)
    ax1.set_xlabel('RE 설비 침투율 (%)')
    ax1.set_ylabel('24시간 총 운영 비용 ($)')
    ax1.set_title('IEEE 39-bus: 침투율 vs 운영 비용')
    ax1.grid(True, alpha=0.3)

    ax2.bar(pens, shares, color='#5A7D6A', width=6)
    ax2.set_xlabel('RE 설비 침투율 (%)')
    ax2.set_ylabel('실제 RE 발전 비중 (%)')
    ax2.set_title('IEEE 39-bus: 침투율 vs RE 발전 비중')
    ax2.grid(True, alpha=0.3, axis='y')

    plt.tight_layout()
    plt.show()
    print("\n침투율이 증가할수록 한계비용 0인 RE가 기존 발전을 대체하여 운영 비용이 감소합니다.")